# Avance 4.4 - Modelos alternativos

**Proyecto academico:** AURORA - Analisis Unificado de Riesgo Operativo y Readiness con Aprendizaje Automatico  
**Equipo:** 4  
**Modalidad:** Individual  
**Tema:** Comparacion, ajuste fino y seleccion de modelos alternativos

## Aviso de privacidad

Este avance utiliza exclusivamente datos sinteticos. Los registros, variables y resultados fueron generados para fines academicos y no contienen datos reales de ninguna organizacion. No se incluyen repositorios, URLs, tecnologias internas, arquitectura, clientes, credenciales, procedimientos operativos ni nombres de sistemas.

## Objetivo del avance

El objetivo de esta libreta es comparar una gama diversa de modelos individuales para predecir la variable `high_risk_change`, seleccionar los dos mejores segun una metrica principal, ajustar sus hiperparametros y justificar la eleccion de un modelo individual final.

La entrega cubre tres puntos centrales de la rubrica:

1. Construccion de al menos seis modelos individuales, no ensambles.
2. Comparacion ordenada por metrica principal, con metricas complementarias y tiempos de entrenamiento.
3. Ajuste fino de los dos mejores modelos y seleccion argumentada del modelo final.

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


def find_repo_root(start: Path) -> Path:
    """Find the academic repository root from common notebook execution locations."""
    starts = [start, *start.parents]
    for candidate in starts:
        direct = candidate / 'entregas' / 'avance_2_feature_engineering' / 'data' / 'aurora_features_model_ready.csv'
        nested = candidate / 'Repositorio_Tec_AURORA' / 'entregas' / 'avance_2_feature_engineering' / 'data' / 'aurora_features_model_ready.csv'
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / 'Repositorio_Tec_AURORA'
    raise FileNotFoundError('No se encontro el repositorio academico AURORA.')

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / 'entregas' / 'avance_4_modelos_alternativos'
DATA_DIR = NOTEBOOK_DIR / 'data'
FIGURES_DIR = DATA_DIR / 'figures'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DATASET = REPO_ROOT / 'entregas' / 'avance_2_feature_engineering' / 'data' / 'aurora_features_model_ready.csv'
BASELINE_METRICS = REPO_ROOT / 'entregas' / 'avance_3_baseline' / 'data' / 'aurora_baseline_metrics.csv'

print('Repositorio academico AURORA localizado correctamente.')
print('Dataset sintetico model-ready cargado desde la entrega previa.')

Repositorio academico AURORA localizado correctamente.
Dataset sintetico model-ready cargado desde la entrega previa.


## Carga del dataset model-ready

Se utiliza el dataset preparado en Avance 2. En esta etapa no se rehace la ingenieria de caracteristicas; el foco esta en comparar modelos y ajustar hiperparametros.

La variable objetivo es `high_risk_change`, que representa si un cambio sintetico requiere atencion de alto riesgo dentro del caso de estudio.

In [2]:
df = pd.read_csv(INPUT_DATASET)

TARGET = 'high_risk_change'
leakage_columns = ['change_id', 'failure_category', 'recommended_validation_scope', 'readiness_score']
present_leakage = [column for column in leakage_columns if column in df.columns]
assert TARGET in df.columns, 'No se encontro la variable objetivo.'
assert not present_leakage, f'Columnas con posible fuga de informacion presentes: {present_leakage}'
assert not df.isna().any().any(), 'El dataset model-ready no debe contener valores faltantes.'

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print('Forma del dataset:', df.shape)
print('Numero de predictores:', X.shape[1])
print('Distribucion de la variable objetivo:')
print(y.value_counts(normalize=True).rename('proporcion').to_frame())

Forma del dataset: (2400, 61)
Numero de predictores: 60
Distribucion de la variable objetivo:
                  proporcion
high_risk_change            
0                       0.72
1                       0.28


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Proporcion clase positiva en train:', round(y_train.mean(), 4))
print('Proporcion clase positiva en test:', round(y_test.mean(), 4))

Train: (1920, 60) Test: (480, 60)
Proporcion clase positiva en train: 0.2802
Proporcion clase positiva en test: 0.2792


## Metricas de evaluacion

La metrica principal sera **PR-AUC / Average Precision**, porque el caso busca identificar cambios de alto riesgo y la clase positiva es minoritaria. En este contexto, una metrica basada solo en exactitud podria ser enganosa: un modelo podria acertar muchos casos de bajo riesgo y aun asi fallar en los casos importantes.

Para evitar usar el conjunto de prueba durante la seleccion de candidatos, la comparativa inicial se realiza con validacion cruzada estratificada sobre el conjunto de entrenamiento. El conjunto de prueba se reserva para evaluar los modelos ajustados.

In [4]:
def positive_scores(model, X_values):
    """Return a continuous score for the positive class when the model supports it."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_values)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X_values)
    return model.predict(X_values)


def evaluate_model(model, X_values, y_true, split, model_name, training_time_sec=None):
    scores = positive_scores(model, X_values)
    y_pred = model.predict(X_values)
    return {
        'model': model_name,
        'split': split,
        'pr_auc_average_precision': average_precision_score(y_true, scores),
        'roc_auc': roc_auc_score(y_true, scores),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_positive': precision_score(y_true, y_pred, zero_division=0),
        'recall_positive': recall_score(y_true, y_pred, zero_division=0),
        'f1_positive': f1_score(y_true, y_pred, zero_division=0),
        'training_time_sec': training_time_sec,
    }


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    'pr_auc_average_precision': 'average_precision',
    'roc_auc': 'roc_auc',
    'balanced_accuracy': 'balanced_accuracy',
    'precision_positive': 'precision',
    'recall_positive': 'recall',
    'f1_positive': 'f1',
}


def cross_validate_candidate(name, model):
    start = time.perf_counter()
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        scoring=SCORING,
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    )
    elapsed = time.perf_counter() - start
    row = {'model': name, 'selection_basis': 'train_stratified_cv'}
    for metric in SCORING:
        values = cv_results[f'test_{metric}']
        row[metric] = values.mean()
        row[f'{metric}_std'] = values.std()
    row['training_time_sec'] = elapsed
    row['mean_fit_time_sec'] = cv_results['fit_time'].mean()
    return row

## Modelos alternativos

Se entrenan seis modelos individuales y variados. No se utilizan ensambles porque la instruccion del avance solicita modelos individuales.

In [5]:
models = {
    'logistic_regression': LogisticRegression(
        max_iter=3000,
        solver='liblinear',
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'svc': SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'knn': KNeighborsClassifier(
        n_neighbors=15,
        weights='distance',
        metric='minkowski',
    ),
    'gaussian_nb': GaussianNB(),
    'decision_tree': DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'mlp_classifier': MLPClassifier(
        hidden_layer_sizes=(32,),
        activation='relu',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=RANDOM_STATE,
    ),
}

comparison_rows = []
for name, model in models.items():
    comparison_rows.append(cross_validate_candidate(name, model))

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        by=['pr_auc_average_precision', 'balanced_accuracy', 'recall_positive'],
        ascending=False,
    )
    .reset_index(drop=True)
)
comparison_df.insert(0, 'rank', np.arange(1, len(comparison_df) + 1))
comparison_df.to_csv(DATA_DIR / 'aurora_alternative_models_comparison.csv', index=False)

comparison_df

,rank,model,selection_basis,pr_auc_average_precision,pr_auc_average_precision_std,roc_auc,roc_auc_std,balanced_accuracy,balanced_accuracy_std,precision_positive,precision_positive_std,recall_positive,recall_positive_std,f1_positive,f1_positive_std,training_time_sec,mean_fit_time_sec
0,1,logistic_regression,train_stratified_cv,0.981606,0.002787,0.992119,0.001045,0.951645,0.004053,0.877464,0.016902,0.955390,0.000117,0.914686,0.009145,1.262716,0.005961
1,2,svc,train_stratified_cv,0.969555,0.000855,0.986713,0.000093,0.932524,0.008613,0.876089,0.036986,0.916408,0.020627,0.895026,0.015656,1.085851,0.017607
2,3,mlp_classifier,train_stratified_cv,0.954298,0.009722,0.979668,0.004912,0.888624,0.022819,0.904587,0.001101,0.810532,0.047613,0.854221,0.026801,0.073942,0.056168
3,4,knn,train_stratified_cv,0.879404,0.018319,0.932540,0.013380,0.721877,0.027213,0.975349,0.003093,0.448096,0.054421,0.612170,0.052981,1.014944,0.001055
4,5,decision_tree,train_stratified_cv,0.767354,0.005248,0.893483,0.006586,0.805961,0.008398,0.625347,0.016762,0.799317,0.039402,0.700724,0.004301,0.076099,0.006795
5,6,gaussian_nb,train_stratified_cv,0.696765,0.056999,0.850463,0.014602,0.751225,0.007984,0.655640,0.021416,0.631968,0.004663,0.643481,0.012780,0.993932,0.001572


In [6]:
if BASELINE_METRICS.exists():
    baseline = pd.read_csv(BASELINE_METRICS)
    baseline_test = baseline[(baseline['model'] == 'logistic_regression_balanced') & (baseline['split'] == 'test')]
    if not baseline_test.empty:
        print('Referencia de Avance 3 - Regresion Logistica baseline:')
        print(baseline_test[['pr_auc_average_precision', 'balanced_accuracy', 'recall_positive', 'f1_positive']].to_string(index=False))
else:
    print('No se encontro el archivo de metricas de Avance 3; se continua con la comparativa de Avance 4.')

Referencia de Avance 3 - Regresion Logistica baseline:
 pr_auc_average_precision  balanced_accuracy  recall_positive  f1_positive
                 0.979969           0.953045         0.955224     0.917563


## Visualizacion comparativa

Las siguientes graficas resumen el rendimiento principal y el tiempo de entrenamiento de la comparativa por validacion cruzada. La primera se ordena por PR-AUC y la segunda ayuda a observar el costo computacional relativo.

In [7]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=comparison_df,
    x='pr_auc_average_precision',
    y='model',
    hue='model',
    palette='viridis',
    legend=False,
)
plt.title('Comparativa de modelos por PR-AUC')
plt.xlabel('PR-AUC / Average Precision')
plt.ylabel('Modelo')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alternative_models_pr_auc.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/alternative_models_pr_auc.png')

plt.figure(figsize=(10, 5))
sns.barplot(
    data=comparison_df.sort_values('training_time_sec', ascending=False),
    x='training_time_sec',
    y='model',
    hue='model',
    palette='mako',
    legend=False,
)
plt.title('Tiempo de entrenamiento por modelo')
plt.xlabel('Segundos')
plt.ylabel('Modelo')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alternative_models_training_time.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/alternative_models_training_time.png')

Figura guardada: data/figures/alternative_models_pr_auc.png
Figura guardada: data/figures/alternative_models_training_time.png


In [8]:
top_two_models = comparison_df.head(2)['model'].tolist()
print('Dos mejores modelos seleccionados para ajuste fino con validacion cruzada en train:', top_two_models)

Dos mejores modelos seleccionados para ajuste fino con validacion cruzada en train: ['logistic_regression', 'svc']


## Ajuste fino de hiperparametros

Se ajustan los dos modelos con mejor desempeno en la comparativa de validacion cruzada. La busqueda se hace con validacion cruzada estratificada y se optimiza PR-AUC, manteniendo consistencia con la metrica principal del problema. El conjunto de prueba se utiliza despues del ajuste para estimar el desempeno final.

In [9]:
model_factories = {
    'logistic_regression': lambda: LogisticRegression(max_iter=4000, solver='liblinear', random_state=RANDOM_STATE),
    'svc': lambda: SVC(random_state=RANDOM_STATE),
    'knn': lambda: KNeighborsClassifier(),
    'gaussian_nb': lambda: GaussianNB(),
    'decision_tree': lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
    'mlp_classifier': lambda: MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE),
}

param_grids = {
    'logistic_regression': {
        'C': [0.01, 0.1, 1.0, 10.0, 50.0],
        'penalty': ['l1', 'l2'],
        'class_weight': [None, 'balanced'],
    },
    'svc': {
        'C': [0.1, 1.0, 5.0, 10.0],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto'],
        'class_weight': [None, 'balanced'],
    },
    'knn': {
        'n_neighbors': [3, 5, 9, 15, 25, 35],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan'],
    },
    'gaussian_nb': {
        'var_smoothing': np.logspace(-12, -7, 6),
    },
    'decision_tree': {
        'max_depth': [3, 5, 8, 12, None],
        'min_samples_leaf': [5, 10, 20, 40],
        'criterion': ['gini', 'entropy'],
        'class_weight': [None, 'balanced'],
    },
    'mlp_classifier': {
        'hidden_layer_sizes': [(16,), (32,), (32, 16)],
        'activation': ['relu', 'tanh'],
        'alpha': [0.0001, 0.001],
        'learning_rate_init': [0.001],
    },
}

tuned_models = {}
tuned_rows = []

for name in top_two_models:
    search = GridSearchCV(
        estimator=model_factories[name](),
        param_grid=param_grids[name],
        scoring='average_precision',
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    start = time.perf_counter()
    search.fit(X_train, y_train)
    tuning_time = time.perf_counter() - start
    tuned_models[name] = search.best_estimator_

    train_metrics = evaluate_model(search.best_estimator_, X_train, y_train, 'train', f'{name}_tuned', tuning_time)
    test_metrics = evaluate_model(search.best_estimator_, X_test, y_test, 'test', f'{name}_tuned', tuning_time)
    for row in [train_metrics, test_metrics]:
        row['base_model'] = name
        row['best_params'] = str(search.best_params_)
        row['best_cv_pr_auc'] = search.best_score_
        tuned_rows.append(row)

    print(f'Modelo ajustado: {name}')
    print('Mejores parametros:', search.best_params_)
    print('Mejor PR-AUC promedio CV:', round(search.best_score_, 4))
    print('Tiempo total de ajuste:', round(tuning_time, 3), 'segundos')
    print('-' * 80)

tuned_df = pd.DataFrame(tuned_rows)
tuned_test_df = (
    tuned_df[tuned_df['split'] == 'test']
    .sort_values(
        by=['pr_auc_average_precision', 'balanced_accuracy', 'recall_positive'],
        ascending=False,
    )
    .reset_index(drop=True)
)
tuned_test_df.to_csv(DATA_DIR / 'aurora_tuned_models_results.csv', index=False)
tuned_test_df

Modelo ajustado: logistic_regression
Mejores parametros: {'C': 1.0, 'class_weight': None, 'penalty': 'l1'}
Mejor PR-AUC promedio CV: 0.982
Tiempo total de ajuste: 0.313 segundos
--------------------------------------------------------------------------------


Modelo ajustado: svc
Mejores parametros: {'C': 0.1, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'linear'}
Mejor PR-AUC promedio CV: 0.9821
Tiempo total de ajuste: 0.82 segundos
--------------------------------------------------------------------------------


,model,split,pr_auc_average_precision,roc_auc,accuracy,balanced_accuracy,precision_positive,recall_positive,f1_positive,training_time_sec,base_model,best_params,best_cv_pr_auc
0,logistic_regression_tuned,test,0.984907,0.993659,0.964583,0.954857,0.939850,0.932836,0.936330,0.313400,logistic_regression,"{'C': 1.0, 'class_weight': None, 'penalty': 'l1'}",0.981995
1,svc_tuned,test,0.984393,0.993637,0.952083,0.957618,0.872483,0.970149,0.918728,0.819934,svc,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'...",0.982127


In [10]:
plt.figure(figsize=(9, 4.5))
sns.barplot(
    data=tuned_test_df,
    x='pr_auc_average_precision',
    y='model',
    hue='model',
    palette='crest',
    legend=False,
)
plt.title('Modelos ajustados por PR-AUC')
plt.xlabel('PR-AUC / Average Precision')
plt.ylabel('Modelo ajustado')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tuned_models_pr_auc.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/tuned_models_pr_auc.png')

Figura guardada: data/figures/tuned_models_pr_auc.png


## Seleccion del modelo individual final

El modelo final se elige con una regla practica orientada al problema. Primero se revisa PR-AUC, pero si los dos modelos ajustados estan practicamente empatados, se priorizan recall de alto riesgo y balanced accuracy. Esta decision es consistente con el caso de estudio: es mas costoso omitir un cambio de alto riesgo que revisar algunos casos adicionales.

Tambien se revisan estabilidad train/test, tiempo de entrenamiento e interpretabilidad. La seleccion no depende de una sola metrica aislada.

In [11]:
train_lookup = tuned_df[tuned_df['split'] == 'train'].set_index('base_model')
selection_rows = []

for _, row in tuned_test_df.iterrows():
    base_name = row['base_model']
    train_row = train_lookup.loc[base_name]
    generalization_gap = train_row['pr_auc_average_precision'] - row['pr_auc_average_precision']
    selection_rows.append({
        'base_model': base_name,
        'tuned_model': row['model'],
        'test_pr_auc': row['pr_auc_average_precision'],
        'test_recall_positive': row['recall_positive'],
        'test_balanced_accuracy': row['balanced_accuracy'],
        'test_precision_positive': row['precision_positive'],
        'test_f1_positive': row['f1_positive'],
        'generalization_gap_pr_auc': generalization_gap,
        'training_time_sec': row['training_time_sec'],
        'best_params': row['best_params'],
    })

selection_df = pd.DataFrame(selection_rows)
max_pr_auc = selection_df['test_pr_auc'].max()
pr_auc_tie_tolerance = 0.005
selection_df['within_pr_auc_tie'] = (max_pr_auc - selection_df['test_pr_auc']) <= pr_auc_tie_tolerance

final_candidate_pool = selection_df[selection_df['within_pr_auc_tie']].copy()
final_candidate_pool = final_candidate_pool.sort_values(
    by=['test_recall_positive', 'test_balanced_accuracy', 'test_f1_positive', 'generalization_gap_pr_auc', 'training_time_sec'],
    ascending=[False, False, False, True, True],
).reset_index(drop=True)

selection_df = selection_df.sort_values(
    by=['within_pr_auc_tie', 'test_pr_auc', 'test_recall_positive', 'test_balanced_accuracy'],
    ascending=[False, False, False, False],
).reset_index(drop=True)

final_base_model = final_candidate_pool.loc[0, 'base_model']
final_model = tuned_models[final_base_model]
final_scores = positive_scores(final_model, X_test)
final_predictions = final_model.predict(X_test)

final_predictions_df = pd.DataFrame({
    'row_id': X_test.index,
    'y_true': y_test.values,
    'risk_score': final_scores,
    'y_pred': final_predictions,
})
final_predictions_df.to_csv(DATA_DIR / 'aurora_final_model_predictions.csv', index=False)

print('Modelo individual final:', final_base_model)
print('Criterio de desempate: PR-AUC dentro de 0.005 se considera empate practico; dentro de ese grupo se prioriza recall positivo y balanced accuracy.')
print('Resumen de seleccion:')
print(selection_df.to_string(index=False))

Modelo individual final: svc
Criterio de desempate: PR-AUC dentro de 0.005 se considera empate practico; dentro de ese grupo se prioriza recall positivo y balanced accuracy.
Resumen de seleccion:
         base_model               tuned_model  test_pr_auc  test_recall_positive  test_balanced_accuracy  test_precision_positive  test_f1_positive  generalization_gap_pr_auc  training_time_sec                                                                  best_params  within_pr_auc_tie
logistic_regression logistic_regression_tuned     0.984907              0.932836                0.954857                 0.939850          0.936330                   0.004633           0.313400                            {'C': 1.0, 'class_weight': None, 'penalty': 'l1'}               True
                svc                 svc_tuned     0.984393              0.970149                0.957618                 0.872483          0.918728                   0.002839           0.819934 {'C': 0.1, 'class_weight': 'ba

In [12]:
cm = confusion_matrix(y_test, final_predictions)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Matriz de confusion - modelo final: {final_base_model}')
plt.xlabel('Prediccion')
plt.ylabel('Valor real')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_confusion_matrix.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_confusion_matrix.png')

precision, recall, _ = precision_recall_curve(y_test, final_scores)
fpr, tpr, _ = roc_curve(y_test, final_scores)

plt.figure(figsize=(6, 4.5))
plt.plot(recall, precision, label='Modelo final')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall del modelo final')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_precision_recall_curve.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_precision_recall_curve.png')

plt.figure(figsize=(6, 4.5))
plt.plot(fpr, tpr, label='Modelo final')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Referencia aleatoria')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC del modelo final')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_roc_curve.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_roc_curve.png')

Figura guardada: data/figures/final_model_confusion_matrix.png
Figura guardada: data/figures/final_model_precision_recall_curve.png
Figura guardada: data/figures/final_model_roc_curve.png


## Interpretacion de la decision final

En esta corrida, los dos modelos ajustados quedan muy cerca en PR-AUC. Por esa razon, la decision final no se toma solamente por la diferencia decimal de esa metrica.

Para el caso de estudio, `svc_tuned` es una opcion mas conveniente porque conserva un PR-AUC practicamente equivalente y mejora el recall de la clase positiva. Esto significa que detecta mas cambios sinteticos de alto riesgo. Tambien obtiene mejor balanced accuracy y menor tiempo de ajuste. Aunque `logistic_regression_tuned` ofrece mayor precision positiva y una interpretacion mas directa, el objetivo principal de esta etapa es no omitir casos de alto riesgo.

El modelo final elegido pasa a la siguiente etapa como candidato principal, con la nota de que debera revisarse su explicabilidad y calibracion en el avance posterior.

In [13]:
final_row = final_candidate_pool.iloc[0]
print('Conclusiones operativas del Avance 4')
print('1. Se compararon seis modelos individuales y variados, sin utilizar ensambles.')
print('2. La comparativa se ordeno por PR-AUC de validacion cruzada en train e incluyo metricas complementarias y tiempo de entrenamiento.')
print('3. Se ajustaron los dos mejores modelos con validacion cruzada estratificada.')
print(f"4. El modelo final seleccionado fue {final_base_model}, con PR-AUC de {final_row['test_pr_auc']:.4f}, recall positivo de {final_row['test_recall_positive']:.4f} y balanced accuracy de {final_row['test_balanced_accuracy']:.4f}.")
print('5. La eleccion prioriza no omitir casos de alto riesgo cuando el PR-AUC esta practicamente empatado entre modelos.')
print('6. Para las siguientes etapas, este modelo puede evaluarse como candidato principal, manteniendo vigilancia sobre calibracion, falsos positivos y explicabilidad.')

Conclusiones operativas del Avance 4
1. Se compararon seis modelos individuales y variados, sin utilizar ensambles.
2. La comparativa se ordeno por PR-AUC de validacion cruzada en train e incluyo metricas complementarias y tiempo de entrenamiento.
3. Se ajustaron los dos mejores modelos con validacion cruzada estratificada.
4. El modelo final seleccionado fue svc, con PR-AUC de 0.9844, recall positivo de 0.9701 y balanced accuracy de 0.9576.
5. La eleccion prioriza no omitir casos de alto riesgo cuando el PR-AUC esta practicamente empatado entre modelos.
6. Para las siguientes etapas, este modelo puede evaluarse como candidato principal, manteniendo vigilancia sobre calibracion, falsos positivos y explicabilidad.


## Conclusion CRISP-ML

En esta etapa se amplio la evaluacion del problema al comparar distintos supuestos de modelado. Los resultados permiten observar que no todos los algoritmos responden igual ante el mismo conjunto de variables, aun cuando el dataset ya esta preparado para modelado.

El ajuste fino de los dos modelos con mejor desempeno ayuda a evitar una seleccion prematura basada en configuraciones por defecto. La eleccion del modelo individual final queda respaldada por metricas, tiempo de entrenamiento, estabilidad e interpretabilidad, pero tambien por el objetivo practico del problema: reducir la probabilidad de omitir cambios de alto riesgo.

Como el caso de estudio utiliza datos sinteticos, las conclusiones no deben interpretarse como resultados reales de una organizacion. Su valor esta en demostrar una metodologia replicable y segura para comparar modelos en un problema de clasificacion de riesgo.